In [10]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve)

# ==============================================================================
# [단계 0] 환경 설정 및 데이터셋 로드 / 전처리
# ==============================================================================
print("="*60)
print("[단계 0] 환경 설정 및 데이터 로드 시작")
print("="*60)

# 시각화 차트를 저장할 'figs' 폴더가 없을 경우 자동 생성
os.makedirs('figs', exist_ok=True)

# Matplotlib 그래프에서 음수(-) 기호가 깨지는 현상 방지
plt.rcParams['axes.unicode_minus'] = False

# 전처리 완료된 머신러닝 데이터셋 CSV 파일 읽기 (한글 인코딩 처리)
df = pd.read_csv('./dataset/processed_ml_dataset.csv', encoding='utf-8-sig')

# 머신러닝 모델의 독립변수(X, 피처)로 사용할 기후 및 작물 생육 조건 컬럼 리스트
feature_cols = ['min_temp', 'max_temp', 'avg_temp', 'avg_rhm', 'annual_rn',
                'opt_temp_min', 'opt_temp_max', 'frost_limit_temp', 'opt_humidity',
                'soil_ph_min', 'soil_ph_max']

# 한글로 된 컬럼명을 영문 컬럼명으로 변경하기 위한 매핑 사전
rename_map = {
    '생육적온_최저(℃)': 'opt_temp_min',
    '생육적온_최고(℃)': 'opt_temp_max',
    '한계생육온도(℃)': 'frost_limit_temp',
    '적정습도(%)': 'opt_humidity',
    '토양pH_최저': 'soil_ph_min',
    '토양pH_최고': 'soil_ph_max',
    '수익성(1-5)': 'profit_score'
}

# 데이터프레임의 컬럼명 일괄 변경
df = df.rename(columns=rename_map)

print(f"✔ 데이터셋 로드 완료: 총 {df.shape[0]}행 {df.shape[1]}열")


# ==============================================================================
# [단계 1] 탐색적 데이터 분석(EDA) 및 시각화 차트 생성 (figs/ 저장)
# ==============================================================================
print("\n" + "="*60)
print("[단계 1] 탐색적 데이터 분석(EDA) 시각화 생성")
print("="*60)

# 상관관계 분석에 활용할 수치형 데이터 컬럼 정의
numeric_cols = ['min_temp', 'max_temp', 'avg_temp', 'avg_rhm', 'annual_rn',
                'opt_temp_min', 'opt_temp_max', 'frost_limit_temp', 'opt_humidity',
                'soil_ph_min', 'soil_ph_max', 'profit_score', 'suitability']

# --- [시각화 1] 피처 간 상관관계 히트맵 (Correlation Heatmap) ---
plt.figure(figsize=(11, 9))
# 피어슨 상관계수를 계산하여 시각화 (소수점 2자리 표시)
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, annot_kws={'size': 8})
plt.title('Correlation Heatmap: Climate x Crop Growth Conditions')
plt.tight_layout()
plt.savefig('figs/01_heatmap.png', dpi=110) # 파일 저장
plt.close() # 메모리 해제
print("✔ [1/7] Correlation Heatmap 저장 완료: figs/01_heatmap.png")

# --- [시각화 2] 수익성 점수 대 작물 재배 적합률 (Profitability vs Suitability) ---
plt.figure(figsize=(7, 5))
# 수익성 등급(1-5)에 따른 평균 적합률 비율을 막대 그래프로 표시
sns.barplot(data=df, x='profit_score', y='suitability', errorbar=None, color='#4C72B0')
plt.title('Profitability Score vs Suitability Rate')
plt.xlabel('Profitability Score (1-5)')
plt.ylabel('Suitability Rate')
plt.tight_layout()
plt.savefig('figs/02_profit_vs_suitability.png', dpi=110)
plt.close()
print("✔ [2/7] Profitability vs Suitability 저장 완료: figs/02_profit_vs_suitability.png")

# --- [시각화 3] 지역별 최저기온 분포 (Regional Min Temperature Distribution) ---
plt.figure(figsize=(7, 5))
# 지역 중복을 제거한 후 국내 98개 지역의 최저기온 분포를 히스토그램 및 밀도 곡선(KDE)으로 시각화
sns.histplot(df.drop_duplicates('region')['min_temp'], bins=20, kde=True, color='#55A868')
plt.title('Regional Minimum Temperature Distribution (98 Korean Regions)')
plt.xlabel('Min Temperature (C)')
plt.tight_layout()
plt.savefig('figs/03_mintemp_dist.png', dpi=110)
plt.close()
print("✔ [3/7] Regional Min Temp Distribution 저장 완료: figs/03_mintemp_dist.png")


# ==============================================================================
# [단계 2] 머신러닝 데이터 분할 및 스케일링
# ==============================================================================
print("\n" + "="*60)
print("[단계 2] 머신러닝 데이터 세트 분할 및 스케일링")
print("="*60)

# 독립변수(X)와 종속변수/타겟(y: 적합 여부 0 또는 1) 추출
X = df[feature_cols].copy()
y = df['suitability']

# 학습용(80%) 및 테스트용(20%) 데이터셋으로 분할 (stratify=y를 적용하여 클래스 불균형 비율 유지)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"✔ 학습 세트 크기: {X_train.shape}, 검증 세트 크기: {X_test.shape}")

# 거리 기반 모델(LogisticRegression)을 위한 표준화 스케일러(StandardScaler) 적용
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train) # 학습 데이터 기반으로 스케일러 학습 및 변환
X_test_s = scaler.transform(X_test)       # 테스트 데이터 변환
print("✔ StandardScaler 적용 완료")


# ==============================================================================
# [단계 3] 머신러닝 모델 비교 학습 및 평가 지표 집계
# ==============================================================================
print("\n" + "="*60)
print("[단계 3] 모델별 비교 학습 및 성능 평가")
print("="*60)

# 비교 실험을 진행할 4가지 주요 분류 알고리즘 객체 생성
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=200),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
}

results = [] # 모델별 평가 지표를 저장할 리스트
roc_data = {} # ROC 커브 그리기를 위한 FPR, TPR 저장 사전

# 반복문을 수행하며 모델 학습 및 예측 진행
for name, model in models.items():
    # 로지스틱 회귀는 스케일링된 데이터 사용, 트리 기반 모델은 원본 피처 데이터 사용
    if name == 'LogisticRegression':
        model.fit(X_train_s, y_train)
        pred = model.predict(X_test_s)
        proba = model.predict_proba(X_test_s)[:, 1] # 양성 클래스(1) 확률값 추출
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        
    f1 = f1_score(y_test, pred, zero_division=0)
    auc = roc_auc_score(y_test, proba)
    print(f"  - [{name}] 학습 완료 -> F1-Score: {f1:.4f}, ROC-AUC: {auc:.4f}")
    
    # 주요 평가 지표 수집
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1,
        'ROC_AUC': auc
    })
    
    # ROC 곡선 좌표 계산
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_data[name] = (fpr, tpr, auc)

# 결과 리스트를 데이터프레임으로 전환 후 F1-score 기준으로 내림차순 정렬
results_df = pd.DataFrame(results).sort_values('F1', ascending=False)

# --- [시각화 4] 모델별 ROC 커브 비교 (ROC Curve Comparison) ---
plt.figure(figsize=(7, 6))
for name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3) # 기준선(Random Line)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig('figs/04_roc_comparison.png', dpi=110)
plt.close()
print("✔ [4/7] ROC Curve Comparison 저장 완료: figs/04_roc_comparison.png")

# --- [시각화 5] 모델별 F1-score 비교 막대 그래프 (F1-score Comparison) ---
plt.figure(figsize=(7, 5))
sns.barplot(data=results_df, x='Model', y='F1', color='#8172B2')
plt.title('F1-score by Model')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('figs/05_f1_comparison.png', dpi=110)
plt.close()
print("✔ [5/7] F1-score Comparison 저장 완료: figs/05_f1_comparison.png")


# ==============================================================================
# [단계 4] 랜덤 포레스트 하이퍼파라미터 튜닝 및 최적 모델 평가
# ==============================================================================
print("\n" + "="*60)
print("[단계 4] 랜덤 포레스트 하이퍼파라미터 튜닝")
print("="*60)

# GridSearchCV로 탐색할 하이퍼파라미터 후보군 설정
param_grid = {
    'n_estimators': [100, 200, 300], # 결정 트리 개수
    'max_depth': [None, 8, 12],       # 트리의 최대 깊이
    'min_samples_leaf': [1, 3, 5]     # 리프 노드에 필요한 최소 샘플 수
}
print(f"  - 탐색 파라미터 조합: {param_grid}")

# 5-Fold 교차 검증을 통해 F1 점수가 가장 높은 최적의 하이퍼파라미터 조합 탐색
grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42), 
    param_grid, 
    cv=5, 
    scoring='f1', 
    n_jobs=-1
)
grid.fit(X_train, y_train)

# 최적 파라미터로 학습된 모델 객체 도출
best_rf = grid.best_estimator_
pred_tuned = best_rf.predict(X_test)
proba_tuned = best_rf.predict_proba(X_test)[:, 1]

print(f"✔ 최적 파라미터: {grid.best_params_}")
print(f"✔ 튜닝 후 성능 -> F1-Score: {f1_score(y_test, pred_tuned):.4f}, ROC-AUC: {roc_auc_score(y_test, proba_tuned):.4f}")

# --- [시각화 6] 튜닝된 랜덤 포레스트의 피처 중요도 (Feature Importance) ---
importances = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(8, 6))
sns.barplot(x=importances.values, y=importances.index, color='#64B5CD')
plt.title('Feature Importance (Tuned RandomForest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('figs/06_feature_importance.png', dpi=110)
plt.close()
print("✔ [6/7] Feature Importance 저장 완료: figs/06_feature_importance.png")

# --- [시각화 7] 튜닝된 모델의 혼동 행렬 (Confusion Matrix) ---
cm = confusion_matrix(y_test, pred_tuned)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Tuned RandomForest)')
plt.tight_layout()
plt.savefig('figs/07_confusion_matrix.png', dpi=110)
plt.close()
print("✔ [7/7] Confusion Matrix 저장 완료: figs/07_confusion_matrix.png")


# ==============================================================================
# [단계 5] 전체 분석 결과 요약 및 종료 메시지
# ==============================================================================
print("\n" + "="*60)
print("[단계 5] 전체 실행 결과 요약")
print("="*60)
print(results_df.to_string(index=False))
print("\n모든 시각화 그래프가 'figs/' 폴더에 성공적으로 저장되었습니다.")

[단계 0] 환경 설정 및 데이터 로드 시작
✔ 데이터셋 로드 완료: 총 9800행 22열

[단계 1] 탐색적 데이터 분석(EDA) 시각화 생성
✔ [1/7] Correlation Heatmap 저장 완료: figs/01_heatmap.png
✔ [2/7] Profitability vs Suitability 저장 완료: figs/02_profit_vs_suitability.png
✔ [3/7] Regional Min Temp Distribution 저장 완료: figs/03_mintemp_dist.png

[단계 2] 머신러닝 데이터 세트 분할 및 스케일링
✔ 학습 세트 크기: (7840, 11), 검증 세트 크기: (1960, 11)
✔ StandardScaler 적용 완료

[단계 3] 모델별 비교 학습 및 성능 평가
  - [LogisticRegression] 학습 완료 -> F1-Score: 0.8354, ROC-AUC: 0.9983
  - [DecisionTree] 학습 완료 -> F1-Score: 0.9811, ROC-AUC: 0.9916
  - [RandomForest] 학습 완료 -> F1-Score: 0.9851, ROC-AUC: 1.0000
  - [GradientBoosting] 학습 완료 -> F1-Score: 0.9850, ROC-AUC: 0.9998
✔ [4/7] ROC Curve Comparison 저장 완료: figs/04_roc_comparison.png
✔ [5/7] F1-score Comparison 저장 완료: figs/05_f1_comparison.png

[단계 4] 랜덤 포레스트 하이퍼파라미터 튜닝
  - 탐색 파라미터 조합: {'n_estimators': [100, 200, 300], 'max_depth': [None, 8, 12], 'min_samples_leaf': [1, 3, 5]}
✔ 최적 파라미터: {'max_depth': 12, 'min_samples_leaf': 1, 'n_estimators': 100}